# Lead Time Check: `commissioningMin`

This notebook is based on `12_leadTimes_example.ipynb` and keeps the setup as close as possible to the original example.

The configuration is the following:
- Two locations: `location1` and `location2`
- One commodity: `commodity1` with unit `unit1`
- Two investment periods: `0` and `1`, each representing one year
- One source `source1` producing `commodity1`
- One sink `sink1` consuming `commodity1`
- `sink1` has no fixed demand in IP 0 and a fixed demand of 1 in IP 1 at both locations
- `source1` has `leadTime = 1`
- `source1` has `commissioningMin = 1` in IP 0 and `commissioningMin = 0` in IP 1

What is checked:
- Whether `commissioningMin` constrains the `commis` variable in the period where it is specified
- Whether the minimum commissioned amount becomes physically available only after the lead time
- Whether the model chooses the minimum amount when this is sufficient to satisfy the IP 1 demand

Expected behaviour with the current lead-time implementation:
- `commissioning` of `source1` should be at least 1 in IP 0
- Because costs are positive and 1 is sufficient, the optimum should choose exactly 1 in IP 0
- `capacity` of `source1` should still be 0 in IP 0
- `capacity` of `source1` should be 1 in IP 1
- `operation_annual` should be 0 in IP 0 and 8760 in IP 1 for each location
- `opexCap` may already appear in IP 0 because the cost logic is still anchored at `commis`

Interpretation:
- If the result follows this pattern, `commissioningMin` applies to construction-start / `commis`, not to physical availability.

Result:
commissioningMin wirkt mit leadTime aktuell als Mindest-Baustart / Mindest-Investitionsstart in der angegebenen IP, nicht als Mindest-Inbetriebnahme in dieser IP.


In [8]:
%load_ext autoreload
%autoreload 2

import fine as fn  # Provides objects and functions to model an energy system
import pandas as pd  # Used to manage data in tables
import numpy as np


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
esM = fn.EnergySystemModel(
    locations = {"location1", "location2"},
    commodities = {"commodity1"},
    commodityUnitsDict = {"commodity1": "unit1"},
    startYear = 0,
    numberOfInvestmentPeriods = 2,
    investmentPeriodInterval = 1
)


In [10]:
esM.add(
    fn.Source(
        esM = esM,
        name = "source1",
        commodity = "commodity1",
        hasCapacityVariable = True,
        operationRateMax = {0: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760)),
                            1: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760))
                            },
        capacityMax = {0: pd.Series({"location1": 100,
                                     "location2": 100}),
                       1: pd.Series({"location1": 100,
                                     "location2": 100})
                       },
        # Test parameter: minimum commissioning in IP 0.
        commissioningMin = {0: pd.Series({"location1": 2,
                                          "location2": 2}),
                            1: pd.Series({"location1": 0,
                                          "location2": 0})
                            },
        investPerCapacity = 1000,
        opexPerCapacity = 1020,
        interestRate = 0.08,
        economicLifetime = 1,
        leadTime = {0: pd.Series({"location1": 1, "location2": 1}),
                    1: pd.Series({"location1": 1, "location2": 1})
                    }
    )
)


In [11]:
esM.add(
    fn.Sink(
        esM = esM,
        name = "sink1",
        commodity = "commodity1",
        hasCapacityVariable = False,
        operationRateFix = {0: pd.DataFrame(np.zeros((8760, 2)), columns=["location1", "location2"], index=range(8760)),
                            1: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760))
                            },
    )
)


In [12]:
solver = fn.utils.ImplementedSolvers.STANDARD_SOLVER.value

esM.optimize(timeSeriesAggregation=False, solver='gurobi')


Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 70092 rows, 70096 columns and 140184 nonzeros (Min)
Model fingerprint: 0x3d665c99
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+03, 2e+03]
  Bounds range     [1e+00, 1e+02]
  RHS range        [0e+00, 0e+00]

Presolve removed 70092 rows and 70096 columns
Presolve time: 0.06s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    8.4000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.09 seconds (0.04 work u

In [13]:
def show_relevant_summary(esM, ip):
    summary = esM.getOptimizationSummary("SourceSinkModel", outputLevel=1, ip=ip)
    relevant_rows = [
        "commissioning", "capacity", "operation", "operation_annual",
        "invest", "capexCap", "opexCap", "capexIfBuilt", "opexIfBuilt",
        "TAC", "NPVcontribution",
    ]
    mask = summary.index.get_level_values(1).isin(relevant_rows)
    display(summary.loc[mask])
    return summary

summary_ip0 = show_relevant_summary(esM, ip=0)
summary_ip1 = show_relevant_summary(esM, ip=1)


location1 location2
Component Property         Unit                            
sink1     NPVcontribution  [1e9 Euro]         0.0       0.0
          TAC              [1e9 Euro/a]       0.0       0.0
          operation        [unit1*h]          0.0       0.0
          operation_annual [unit1*h/a]        0.0       0.0
source1   NPVcontribution  [1e9 Euro]      4200.0    4200.0
          TAC              [1e9 Euro/a]    4200.0    4200.0
          capacity         [unit1]            0.0       0.0
          capexCap         [1e9 Euro/a]    2160.0    2160.0
          commissioning    [unit1]            2.0       2.0
          invest           [1e9 Euro]      2000.0    2000.0
          operation        [unit1*h]          0.0       0.0
          operation_annual [unit1*h/a]        0.0       0.0
          opexCap          [1e9 Euro/a]    2040.0    2040.0

location1 location2
Component Property         Unit                            
sink1     NPVcontribution  [1e9 Euro]         0.0       0.0
          TAC              [1e9 Euro/a]       0.0       0.0
          operation        [unit1*h]       8760.0    8760.0
          operation_annual [unit1*h/a]     8760.0    8760.0
source1   NPVcontribution  [1e9 Euro]         0.0       0.0
          TAC              [1e9 Euro/a]       0.0       0.0
          capacity         [unit1]            2.0       2.0
          capexCap         [1e9 Euro/a]       0.0       0.0
          commissioning    [unit1]            0.0       0.0
          invest           [1e9 Euro]         0.0       0.0
          operation        [unit1*h]       8760.0    8760.0
          operation_annual [unit1*h/a]     8760.0    8760.0
          opexCap          [1e9 Euro/a]       0.0       0.0